# 02 – Information Agent (Metrics & Anomaly Detection)

The **InformationAgent** queries the Databricks/SQL layer for metrics across four data products:
`retention`, `bookings`, `cac`, `ltv`. It also runs anomaly detection against configured thresholds.

**Mock mode enabled** — no Databricks credentials needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from agents.information_agent import InformationAgent
from core.base_agent import AgentRequest

agent = InformationAgent()

## 1. Query a single product

In [ ]:
req = AgentRequest(
    query='What is the current retention rate?',
    data_products=['retention'],
    time_range='last_30_days',
)
result = agent.execute(req)

print('Success  :', result.success)
print('Message  :', result.message)
print('Confidence:', result.confidence)

metrics = result.data['metrics']['retention']
print('\nRetention metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 2. Anomaly detection — normal data (no alerts)

In [ ]:
result = agent.execute(AgentRequest(query='retention metrics', data_products=['retention']))
anomalies = result.data.get('anomalies', [])
print('Anomalies detected:', anomalies if anomalies else 'None — all metrics within thresholds')

## 3. Anomaly detection — low GRR (triggers alert)

In [ ]:
from services.databricks.mock import MockDatabricksService

# Use the low-GRR mock (GRR=78%, below 85% threshold)
low_grr_agent = InformationAgent(data_service=MockDatabricksService(low_grr=True))
result = low_grr_agent.execute(AgentRequest(query='retention metrics', data_products=['retention']))

print('Anomalies:')
for a in result.data.get('anomalies', []):
    print('  WARNING:', a)

## 4. Multi-product query

In [ ]:
req_all = AgentRequest(
    query='Give me a full overview of all data products',
    data_products=['retention', 'bookings', 'cac', 'ltv'],
)
result = agent.execute(req_all)

print(f"Products queried: {result.metadata['products_queried']}")
print(f"Anomaly count: {result.metadata['anomaly_count']}")
print(f"Execution time: {result.execution_time_ms:.1f}ms\n")

for product, metrics in result.data['metrics'].items():
    print(f'--- {product.upper()} ---')
    for k, v in metrics.items():
        print(f'  {k}: {v}')

## 5. Auto product detection from query keywords

In [ ]:
queries = [
    'What is the GRR trend?',
    'How is our ARR performing?',
    'Show me CAC payback period',
    "What's the LTV:CAC ratio?",
]

for q in queries:
    r = agent.execute(AgentRequest(query=q))
    products = r.metadata['products_queried']
    print(f"Query: '{q}'  =>  products detected: {products}")

## 6. Anomaly thresholds reference

In [ ]:
from agents.information_agent import _THRESHOLDS

print('Configured anomaly thresholds:')
for product, rules in _THRESHOLDS.items():
    print(f'  {product}: {rules}')